<a href="https://colab.research.google.com/github/syedbeeban/IE/blob/main/05_L_Curve_Method_for_Regularization_Parameter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This is the exact right approach to take for an SCI-indexed journal. Reviewers are often highly skeptical of "hand-picked" hyperparameters, especially in inverse problems. By using the **L-curve method**, you provide a mathematically rigorous, objective justification for your choice of the regularization parameter ($\alpha$).

The L-curve is a log-log plot that visualizes the trade-off between two competing forces:

1. **The Residual Norm (Data Misfit):** How well the model fits the noisy sensor data, typically represented as $||Kf - g||_2^2$.
2. **The Solution Norm (Regularization Penalty):** The complexity or roughness of the solution, typically represented as $||f||_2^2$ (for standard Tikhonov) or $||\nabla f||_2^2$ (for Sobolev).

When you plot these against each other for various values of $\alpha$, the graph forms a distinct "L" shape.

* The vertical part of the "L" represents under-regularized solutions (fitting the noise).
* The horizontal part represents over-regularized solutions (ignoring the data, too flat).
* **The optimal $\alpha$ is located exactly at the "corner" (the point of maximum curvature).**

Here is the executable Python code that models an ill-posed Fredholm integral equation of the first kind, generates the L-curve, and mathematically calculates the point of maximum curvature to find the exact optimal $\alpha$.

### Python Implementation of the L-Curve Method

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import UnivariateSpline

# 1. Setup the Ill-Posed Problem (Fredholm 1st Kind)
n_points = 100
t = np.linspace(0, 1, n_points)
s = np.linspace(0, 1, n_points)
dt = t[1] - t[0]

# Define a smoothing Kernel (Gaussian blur) - this causes the information loss
S, T = np.meshgrid(s, t, indexing='ij')
K_matrix = np.exp(-10 * (S - T)**2) * dt

# Define the TRUE source function (what we want to find)
true_f = np.sin(np.pi * t) + 0.5 * np.sin(3 * np.pi * t)

# Calculate EXACT sensor data, then add NOISE to make it an inverse problem
exact_g = K_matrix @ true_f
np.random.seed(42)
noise_level = 0.05  # 5% noise
noisy_g = exact_g + noise_level * np.max(np.abs(exact_g)) * np.random.randn(n_points)

# 2. L-Curve Generation
# Define a wide range of alpha values logarithmically
alphas = np.logspace(-6, 1, 100)
residual_norms = []
solution_norms = []

for alpha in alphas:
    # Solve Tikhonov Regularization: (K^T * K + alpha * I) * f = K^T * g
    # This is the discrete matrix equivalent of the integral penalty
    H = K_matrix.T @ K_matrix + alpha * np.eye(n_points)
    rhs = K_matrix.T @ noisy_g

    # Calculate the regularized solution for this specific alpha
    f_alpha = np.linalg.solve(H, rhs)

    # Calculate Residual Norm (Data Misfit) ||Kf - g||^2
    res_norm = np.sum((K_matrix @ f_alpha - noisy_g)**2)
    residual_norms.append(res_norm)

    # Calculate Solution Norm (Penalty) ||f||^2
    sol_norm = np.sum(f_alpha**2)
    solution_norms.append(sol_norm)

# Convert to log scale for the L-curve
log_res = np.log(residual_norms)
log_sol = np.log(solution_norms)

# 3. Mathematically Find the Corner (Maximum Curvature)
# We use splines to create continuous functions of the curve,
# allowing us to calculate the first and second derivatives mathematically.
spline_res = UnivariateSpline(np.log10(alphas), log_res, s=1)
spline_sol = UnivariateSpline(np.log10(alphas), log_sol, s=1)

# Derivatives with respect to log10(alpha)
res_prime = spline_res.derivative(1)(np.log10(alphas))
res_double_prime = spline_res.derivative(2)(np.log10(alphas))

sol_prime = spline_sol.derivative(1)(np.log10(alphas))
sol_double_prime = spline_sol.derivative(2)(np.log10(alphas))

# Curvature formula for parametric curves: k = |x'y'' - y'x''| / (x'^2 + y'^2)^(3/2)
curvature = np.abs(res_prime * sol_double_prime - sol_prime * res_double_prime) / \
            (res_prime**2 + sol_prime**2)**(1.5)

# The optimal alpha is the one that maximizes the curvature
optimal_idx = np.argmax(curvature)
optimal_alpha = alphas[optimal_idx]

print(f"Mathematical Proof Complete.")
print(f"Optimal Regularization Parameter (alpha) identified at: {optimal_alpha:.2e}")

# 4. (Optional) Plotting the results for your paper
plt.figure(figsize=(10, 5))

# Plot 1: The L-Curve
plt.subplot(1, 2, 1)
plt.plot(log_res, log_sol, 'b-', label='L-Curve')
plt.plot(log_res[optimal_idx], log_sol[optimal_idx], 'ro', markersize=8,
         label=f'Optimal Corner ($\\alpha$={optimal_alpha:.2e})')
plt.xlabel('Log Residual Norm $\\log(||Kf - g||^2)$')
plt.ylabel('Log Solution Norm $\\log(||f||^2)$')
plt.title('The L-Curve')
plt.legend()
plt.grid(True)

# Plot 2: Curvature vs Alpha
plt.subplot(1, 2, 2)
plt.semilogx(alphas, curvature, 'g-', label='Curvature $\\kappa$')
plt.axvline(optimal_alpha, color='r', linestyle='--', label='Max Curvature')
plt.xlabel('Regularization Parameter ($\\alpha$)')
plt.ylabel('Curvature')
plt.title('Curvature as a Function of $\\alpha$')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

### How to Frame This in Your Paper

When writing the methodology section for your SCI paper, you can use this workflow to construct a compelling narrative:

1. **State the Problem:** Point out that Neural Networks solving Fredholm integral equations of the first kind require a regularization parameter $\alpha$ to balance the data loss and the physics/smoothness penalty.
2. **Highlight the Flaw in Prior Work:** Mention that many studies choose this parameter heuristically or via trial and error, which lacks mathematical guarantee.
3. **Introduce Your Approach:** State that you utilize the L-curve criterion to automate the selection of $\alpha$.
4. **Provide the Equation:** Include the curvature formula in your paper:

$$\kappa(\alpha) = \frac{|\eta' \rho'' - \rho' \eta''|}{((\eta')^2 + (\rho')^2)^{3/2}}$$



where $\eta = \log(||Kf - g||_2^2)$ and $\rho = \log(||\nabla f||_2^2)$. Mention that the network is parameterized using the $\alpha$ that yields $\max(\kappa(\alpha))$.

This code extracts the optimal $\alpha$ using classical matrix discretization. Because neural network training is highly stochastic, it is standard practice to compute the optimal $\alpha$ precisely using this discretized method, and then directly plug that proven $\alpha$ into your Neural Network loss function for the deep learning portion of the experiment.

Would you like me to draft a formalized mathematical paragraph containing this logic that you can adapt directly into the methodology section of your manuscript?